# Análisis de cobertura del inventario de repuestos

**Proyecto:** gestión de mantenimiento de equipos portuarios  
**Objetivo:** relacionar las existencias actuales con los movimientos históricos de repuestos para identificar riesgos de desabastecimiento, niveles adecuados y sobrestock.

La cobertura se calcula como:

`cantidad disponible / consumo promedio mensual`

Para este análisis, los movimientos positivos del consolidado se interpretan como consumo o utilización histórica de repuestos.


## 1. Librerías, rutas y nombres de informes


In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display


def localizar_carpeta_etl():
    actual = Path.cwd().resolve()
    candidatos = []
    for base in (actual, *actual.parents):
        candidatos.extend((base, base / "ETL"))

    for carpeta in candidatos:
        if (carpeta / "Data").is_dir() and (carpeta / "src").is_dir():
            return carpeta

    rutas = "\n- ".join(str(ruta) for ruta in candidatos)
    raise FileNotFoundError("No se encontró la carpeta ETL. Rutas revisadas:\n- " + rutas)


ETL_DIR = localizar_carpeta_etl()
DATA_DIR = ETL_DIR / "Data"
RESULTADOS_DIR = ETL_DIR / "Resultado"
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

ruta_inventario = DATA_DIR / "Inventario.xlsx"
ruta_movimientos = RESULTADOS_DIR / (
    "Informe_Consolidado_Movimientos_Repuestos_Mantenimiento_Portuario.xlsx"
)
ruta_salida = RESULTADOS_DIR / (
    "Informe_Analisis_Cobertura_Inventario_Repuestos_Mantenimiento_Portuario.xlsx"
)
ruta_sin_movimientos = DATA_DIR / (
    "Informe_Repuestos_Sin_Movimientos_Mantenimiento_Portuario.xlsx"
)
ruta_repuestos_criticos = RESULTADOS_DIR / (
    "Informe_Repuestos_Criticos_Mantenimiento_Portuario.xlsx"
)

candidatos_maestro = [
    DATA_DIR / "Informe_Maestro_Repuestos_Mantenimiento_Equipos_Portuarios.xlsx",
    DATA_DIR / "Tabla_Maestra_Repuestos.xlsx",
    DATA_DIR / "Tabla_Maestra_Repuesto.xlsx",
]
ruta_maestro = next((ruta for ruta in candidatos_maestro if ruta.is_file()), None)

# Compatibilidad temporal con el consolidado anterior.
if not ruta_movimientos.is_file():
    ruta_anterior = RESULTADOS_DIR / "Ventas_Limpias_PowerBI.xlsx"
    if ruta_anterior.is_file():
        ruta_movimientos = ruta_anterior

requeridos = [ruta_inventario, ruta_movimientos]
faltantes = [str(ruta) for ruta in requeridos if not ruta.is_file()]
if ruta_maestro is None:
    faltantes.append("tabla maestra de repuestos")
if faltantes:
    raise FileNotFoundError("No se encontraron insumos requeridos:\n- " + "\n- ".join(faltantes))

print(f"Inventario: {ruta_inventario.name}")
print(f"Movimientos: {ruta_movimientos.name}")
print(f"Tabla maestra: {ruta_maestro.name}")


## 2. Funciones de normalización y carga


In [ ]:
def normalizar_texto(valor):
    if pd.isna(valor):
        return ""
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    return re.sub(r"\s+", " ", texto).strip().upper()


def encabezados_unicos(columnas):
    resultado = []
    conteos = {}
    for posicion, columna in enumerate(columnas):
        nombre = normalizar_texto(columna).replace(" ", "_")
        nombre = nombre or f"COLUMNA_SIN_NOMBRE_{posicion + 1}"
        conteos[nombre] = conteos.get(nombre, 0) + 1
        if conteos[nombre] > 1:
            nombre = f"{nombre}_{conteos[nombre]}"
        resultado.append(nombre)
    return resultado


def normalizar_codigo(serie):
    codigos = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    numericos = codigos.str.fullmatch(r"\d+", na=False)
    codigos.loc[numericos] = codigos.loc[numericos].str.zfill(4)
    return codigos


def leer_hoja_con_columnas(ruta, grupos_columnas):
    """Busca la primera hoja que contenga al menos una columna de cada grupo."""
    libro = pd.ExcelFile(ruta)
    for hoja in libro.sheet_names:
        df = pd.read_excel(ruta, sheet_name=hoja)
        df.columns = encabezados_unicos(df.columns)
        if all(any(columna in df.columns for columna in grupo) for grupo in grupos_columnas):
            return df, hoja
    raise ValueError(f"No se encontró una hoja compatible en '{ruta.name}'.")


## 3. Limpieza del inventario disponible


In [ ]:
df_inventario_origen, hoja_inventario = leer_hoja_con_columnas(
    ruta_inventario,
    [
        ("NOMBRE_EN_PANTALLA", "REPUESTO", "PRODUCTO"),
        ("CANTIDAD_DISPONIBLE_PARA_USO", "CANTIDAD_DISPONIBLE", "EXISTENCIA"),
    ],
)

columna_detalle = next(
    columna
    for columna in ("NOMBRE_EN_PANTALLA", "REPUESTO", "PRODUCTO")
    if columna in df_inventario_origen.columns
)
columna_cantidad = next(
    columna
    for columna in (
        "CANTIDAD_DISPONIBLE_PARA_USO",
        "CANTIDAD_DISPONIBLE",
        "EXISTENCIA",
    )
    if columna in df_inventario_origen.columns
)

detalle = (
    df_inventario_origen[columna_detalle]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)
df_inventario = pd.DataFrame(
    {
        "CODIGO_REPUESTO": normalizar_codigo(
            detalle.str.extract(r"\[([^\]]+)\]", expand=False)
        ),
        "NOMBRE_REPUESTO_INVENTARIO": (
            detalle.str.replace(r"^\s*\[[^\]]+\]\s*", "", regex=True)
        ),
        "CANTIDAD_DISPONIBLE": pd.to_numeric(
            df_inventario_origen[columna_cantidad],
            errors="coerce",
        ),
    }
)
df_inventario = df_inventario.dropna(
    subset=["CODIGO_REPUESTO", "CANTIDAD_DISPONIBLE"]
)

# Si el inventario repite un código, se suman sus existencias.
df_inventario = (
    df_inventario.groupby("CODIGO_REPUESTO", as_index=False)
    .agg(
        NOMBRE_REPUESTO_INVENTARIO=("NOMBRE_REPUESTO_INVENTARIO", "first"),
        CANTIDAD_DISPONIBLE=("CANTIDAD_DISPONIBLE", "sum"),
    )
)

if df_inventario.empty:
    raise ValueError("El archivo de inventario no produjo registros válidos.")

print(f"Hoja de inventario: {hoja_inventario}")
print(f"Repuestos con existencia registrada: {len(df_inventario):,}")
print(f"Registros con cantidad negativa: {(df_inventario['CANTIDAD_DISPONIBLE'] < 0).sum():,}")
display(df_inventario.head())


## 4. Carga de movimientos históricos y tabla maestra


In [ ]:
df_movimientos, hoja_movimientos = leer_hoja_con_columnas(
    ruta_movimientos,
    [
        ("CODIGO_REPUESTO", "CODIGO_PRODUCTO"),
        ("PERIODO",),
        ("CANTIDAD_MOVIMIENTO", "CANTIDAD_FACTURADA"),
    ],
)
df_movimientos = df_movimientos.rename(
    columns={
        "CODIGO_PRODUCTO": "CODIGO_REPUESTO",
        "NOMBRE_PRODUCTO": "NOMBRE_REPUESTO",
        "CANTIDAD_FACTURADA": "CANTIDAD_MOVIMIENTO",
    }
)
df_movimientos["CODIGO_REPUESTO"] = normalizar_codigo(
    df_movimientos["CODIGO_REPUESTO"]
)
df_movimientos["PERIODO"] = df_movimientos["PERIODO"].map(normalizar_texto)
df_movimientos["CANTIDAD_MOVIMIENTO"] = pd.to_numeric(
    df_movimientos["CANTIDAD_MOVIMIENTO"],
    errors="coerce",
)
df_movimientos = df_movimientos[
    df_movimientos["CODIGO_REPUESTO"].notna()
    & df_movimientos["PERIODO"].ne("")
    & (df_movimientos["CANTIDAD_MOVIMIENTO"] > 0)
].copy()

df_maestro, hoja_maestro = leer_hoja_con_columnas(
    ruta_maestro,
    [
        ("CODIGO_REPUESTO", "CODIGO_PRODUCTO"),
        ("NOMBRE_REPUESTO", "NOMBRE_PRODUCTO"),
        ("CLASIFICACION_I",),
        ("CLASIFICACION_II",),
        ("CLASIFICACION_III",),
        ("CLASIFICACION_IV",),
    ],
)
df_maestro = df_maestro.rename(
    columns={
        "CODIGO_PRODUCTO": "CODIGO_REPUESTO",
        "NOMBRE_PRODUCTO": "NOMBRE_REPUESTO",
    }
)
df_maestro["CODIGO_REPUESTO"] = normalizar_codigo(df_maestro["CODIGO_REPUESTO"])
df_maestro = df_maestro.drop_duplicates("CODIGO_REPUESTO", keep="first")

print(f"Hoja de movimientos: {hoja_movimientos}; registros: {len(df_movimientos):,}")
print(f"Hoja de tabla maestra: {hoja_maestro}; repuestos: {len(df_maestro):,}")


## 5. Consumo mensual y cobertura del inventario


In [ ]:
df_consumo_mensual = (
    df_movimientos.groupby(["CODIGO_REPUESTO", "PERIODO"], as_index=False)
    .agg(CANTIDAD_CONSUMIDA=("CANTIDAD_MOVIMIENTO", "sum"))
)

df_indicadores_consumo = (
    df_consumo_mensual.groupby("CODIGO_REPUESTO", as_index=False)
    .agg(
        CONSUMO_PROMEDIO_MENSUAL=("CANTIDAD_CONSUMIDA", "mean"),
        CONSUMO_MAXIMO_MENSUAL=("CANTIDAD_CONSUMIDA", "max"),
        CONSUMO_MINIMO_MENSUAL=("CANTIDAD_CONSUMIDA", "min"),
        MESES_CON_MOVIMIENTO=("PERIODO", "nunique"),
        CONSUMO_TOTAL_HISTORICO=("CANTIDAD_CONSUMIDA", "sum"),
    )
)

columnas_maestro = [
    "CODIGO_REPUESTO",
    "NOMBRE_REPUESTO",
    "CLASIFICACION_I",
    "CLASIFICACION_II",
    "CLASIFICACION_III",
    "CLASIFICACION_IV",
]

df_cobertura = (
    df_inventario.merge(
        df_indicadores_consumo,
        on="CODIGO_REPUESTO",
        how="left",
        validate="one_to_one",
    )
    .merge(
        df_maestro[columnas_maestro],
        on="CODIGO_REPUESTO",
        how="left",
        validate="one_to_one",
    )
)

df_cobertura["MESES_COBERTURA"] = np.where(
    df_cobertura["CONSUMO_PROMEDIO_MENSUAL"] > 0,
    df_cobertura["CANTIDAD_DISPONIBLE"]
    / df_cobertura["CONSUMO_PROMEDIO_MENSUAL"],
    np.nan,
)
df_cobertura["MESES_COBERTURA"] = df_cobertura["MESES_COBERTURA"].round(2)


def clasificar_inventario(fila):
    cantidad = fila["CANTIDAD_DISPONIBLE"]
    meses = fila["MESES_COBERTURA"]
    if cantidad < 0:
        return "REVISAR INVENTARIO NEGATIVO"
    if cantidad == 0 and pd.notna(meses):
        return "AGOTADO"
    if pd.isna(meses):
        return "SIN MOVIMIENTOS HISTORICOS"
    if meses <= 1:
        return "CRITICO"
    if meses <= 2:
        return "BAJO"
    if meses <= 4:
        return "ADECUADO"
    if meses <= 8:
        return "ALTO"
    return "SOBRESTOCK"


df_cobertura["ESTADO_INVENTARIO"] = df_cobertura.apply(
    clasificar_inventario,
    axis=1,
)

orden_columnas = [
    "CODIGO_REPUESTO",
    "NOMBRE_REPUESTO",
    "NOMBRE_REPUESTO_INVENTARIO",
    "CLASIFICACION_I",
    "CLASIFICACION_II",
    "CLASIFICACION_III",
    "CLASIFICACION_IV",
    "CANTIDAD_DISPONIBLE",
    "CONSUMO_PROMEDIO_MENSUAL",
    "CONSUMO_MAXIMO_MENSUAL",
    "CONSUMO_MINIMO_MENSUAL",
    "MESES_CON_MOVIMIENTO",
    "CONSUMO_TOTAL_HISTORICO",
    "MESES_COBERTURA",
    "ESTADO_INVENTARIO",
]
df_cobertura = (
    df_cobertura[orden_columnas]
    .sort_values(["ESTADO_INVENTARIO", "MESES_COBERTURA", "NOMBRE_REPUESTO"])
    .reset_index(drop=True)
)

display(df_cobertura.head())


## 6. Controles e informes auxiliares


In [ ]:
df_sin_movimientos = df_cobertura[
    df_cobertura["ESTADO_INVENTARIO"] == "SIN MOVIMIENTOS HISTORICOS"
].copy()
df_criticos = df_cobertura[
    df_cobertura["ESTADO_INVENTARIO"].isin(
        ["AGOTADO", "CRITICO", "REVISAR INVENTARIO NEGATIVO"]
    )
].copy()

with pd.ExcelWriter(ruta_sin_movimientos, engine="openpyxl") as writer:
    df_sin_movimientos.to_excel(
        writer,
        sheet_name="Sin_Movimientos",
        index=False,
    )

with pd.ExcelWriter(ruta_repuestos_criticos, engine="openpyxl") as writer:
    df_criticos.to_excel(
        writer,
        sheet_name="Repuestos_Criticos",
        index=False,
    )

print(f"Repuestos sin movimientos: {len(df_sin_movimientos):,}")
print(f"Repuestos agotados, críticos o con inventario negativo: {len(df_criticos):,}")


## 7. Informe consolidado de cobertura


In [ ]:
resumen_estado = (
    df_cobertura.groupby("ESTADO_INVENTARIO", as_index=False)
    .agg(
        REPUESTOS=("CODIGO_REPUESTO", "count"),
        INVENTARIO_TOTAL=("CANTIDAD_DISPONIBLE", "sum"),
    )
    .sort_values("ESTADO_INVENTARIO")
)

indicadores_generales = pd.DataFrame(
    {
        "INDICADOR_MANTENIMIENTO": [
            "Repuestos analizados",
            "Unidades disponibles",
            "Repuestos agotados",
            "Repuestos críticos",
            "Repuestos sin movimientos históricos",
        ],
        "VALOR": [
            len(df_cobertura),
            df_cobertura["CANTIDAD_DISPONIBLE"].sum(),
            (df_cobertura["ESTADO_INVENTARIO"] == "AGOTADO").sum(),
            (df_cobertura["ESTADO_INVENTARIO"] == "CRITICO").sum(),
            len(df_sin_movimientos),
        ],
    }
)

with pd.ExcelWriter(ruta_salida, engine="openpyxl") as writer:
    df_cobertura.to_excel(writer, sheet_name="Cobertura_Repuestos", index=False)
    resumen_estado.to_excel(writer, sheet_name="Resumen_Estados", index=False)
    indicadores_generales.to_excel(
        writer,
        sheet_name="Indicadores_Mantenimiento",
        index=False,
    )
    df_consumo_mensual.to_excel(
        writer,
        sheet_name="Consumo_Mensual",
        index=False,
    )

print(f"Informe de cobertura generado: {ruta_salida}")
display(indicadores_generales)
display(resumen_estado)


## Resultado

El notebook genera:

- `Informe_Analisis_Cobertura_Inventario_Repuestos_Mantenimiento_Portuario.xlsx`
- `Informe_Repuestos_Criticos_Mantenimiento_Portuario.xlsx`
- `Informe_Repuestos_Sin_Movimientos_Mantenimiento_Portuario.xlsx`

Los informes permiten priorizar compras, revisar inventario agotado o negativo y detectar repuestos con exceso de cobertura para la gestión de mantenimiento de equipos portuarios.
